## This program uses a fine-tuned Mistral model to detect flaws in Python code and produce refactored versions of the code.

In [ ]:
# ----------------------------------------------------------------------------
# 
# Please put your program in Test1.py.
# The result will be saved in Results.txt

# Full_Model=True     for using a full model
# Full_Model=False    for using LoRA adapter

# choose 1 for version 1 or 2 for version 2.
# choose 1 for Flaws or 2 for Refactoring in "Flaw_Refactor" variable.
# ----------------------------------------------------------------------------
FilePath="Test1.py"
Results = "Results.txt"
# ----------------------------------------------------------------------------
Full_Model=False
Version=1
Flaw_Refactor=2

if Flaw_Refactor==2: # There is one version for refactoring
   Version=1

In [ ]:
import time
import torch
import os
import re
import ast

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from huggingface_hub import login

In [ ]:
DEVICE="cuda:0" if torch.cuda.is_available() else "cpu"

In [ ]:
token = "YOUR_TOKEN_HERE" 

login(token=token)

In [ ]:
# ----------------------------------------------------------------------------

if Full_Model==False:
   if Flaw_Refactor==1: 
      if Version==1: 
         CheckPoint='HA-Siala/Detect-Flaws-v0.1'       
      else:    
         CheckPoint='HA-Siala/Detect-Flaws-v0.2' 
   else:
      CheckPoint='HA-Siala/RefactoringPy-v0.1'
       
   mistral_checkpoint = "mistralai/Mistral-7B-v0.3"
   tokenizer = AutoTokenizer.from_pretrained(mistral_checkpoint, use_fast=True) 
   tokenizer.pad_token = tokenizer.unk_token   
   tokenizer.padding_side = "left" 
   model = AutoModelForCausalLM.from_pretrained(mistral_checkpoint, torch_dtype=torch.bfloat16, device_map="auto") 
   model = PeftModel.from_pretrained(model, CheckPoint, torch_dtype=torch.bfloat16, is_trainable=False) 
else:
   if Flaw_Refactor==1: 
      if Version==1: 
         CheckPoint='HA-Siala/Detect-Flaws-full-v0.1'       
      else:    
         CheckPoint='HA-Siala/Detect-Flaws-full-v0.2' 
   else:
      CheckPoint='HA-Siala/RefactoringPy-full-v0.1'
    
   model = AutoModelForCausalLM.from_pretrained(CheckPoint, torch_dtype=torch.bfloat16, device_map="auto", low_cpu_mem_usage=True) 
   tokenizer = AutoTokenizer.from_pretrained(CheckPoint, use_fast=True)
   tokenizer.pad_token = tokenizer.unk_token   
   tokenizer.padding_side = "left" 
model.eval() 
# ----------------------------------------------------------------------------

In [ ]:
# ----------------------------------------------------------------------------
def format_time(seconds):
   h = int(seconds // 3600)
   m = int((seconds % 3600) // 60)
   s = seconds % 60
   result = ""
   if h > 0:
      result += f"{h}h "
   if m > 0 or h > 0:  
      result += f"{m}m "
   result += f"{s:.6f}s"
   return result
# ----------------------------------------------------------------------------   
def GeneratePrompt(content):
   Instruction="""Analyze the following Python code, identify any flaws, with a short explanation, and give one or more corrected versions of the code, including a brief explanation of each correction approach:"""
   prompt = f"""Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately solves the following Task:

### Instruction:
{Instruction}

### Code:
{content}
### Response:
"""
   return prompt
# ----------------------------------------------------------------------------

In [ ]:
# ----------------------------------------------------------------------------
def GenerateInferenceOutput(text): 
   tokenizer.pad_token = tokenizer.unk_token  
   inputs = tokenizer(GeneratePrompt(text), return_tensors="pt").to(DEVICE)
    
   InputTokens = inputs["input_ids"].shape[1] 
   torch.cuda.empty_cache() # new 
   with torch.inference_mode():
      outputs = model.generate(
         **inputs,  
         max_new_tokens= 32768, 
         temperature= 0.0, # 0.2, 
         do_sample= False, # True, 
         pad_token_id=tokenizer.eos_token_id,
         top_p= 1.0, # 0.9,
      )
      output_p = tokenizer.batch_decode(outputs, skip_special_tokens=True)

      if output_p:
         output_text = output_p[0]
         split_text = output_text.split("Response:")
         if len(split_text) > 1:             
            return split_text[1].strip(), InputTokens 
         else: 
            return None, InputTokens 
      else:
         return None,0
# ---------------------------------------------------------------------------- 
def ExtractCleanDict(text):
   text = str(text)
   if "### Response:" in text:
      text = text.split("### Response:")[-1].strip()
   else:
      text = text.strip() 
   match = re.search(r"(\{'Flaws'[\s\S]*\})", text)
   if not match:
      raise ValueError("No dict found")
   candidate = match.group(1)
   candidate = candidate.replace("\n", "\\n")
   return ast.literal_eval(candidate)

In [ ]:
# ----------------------------------------------------------------------------
def ReadFile():
   if not os.path.exists(FilePath):
      print(f"Error: File '{FilePath}' does not exist.")
      return None
   if not FilePath.endswith('.py'):
      print(f"Warning: '{FilePath}' may not be a text file.")
   with open(FilePath, 'r', encoding='utf-8') as file:
      content = file.read()
      return content

content = ReadFile()
if content:
   print("Successfully read file!")
   print() 
   StartTime=time.time()
   Output, InputTokens = GenerateInferenceOutput(content)
   Output1_dict = ExtractCleanDict(Output)
   EndTime=time.time()
   InferenceTime=EndTime - StartTime
   TimePerInputToken = InferenceTime / InputTokens
   with open(Results, "w", encoding="utf-8") as target_file:
      target_file.write("Flaws:\n")
      for entry in Output1_dict['Flaws']:
         target_file.write(f"   - {entry['Flaw']}: {entry['Explanation']}\n")
      if Flaw_Refactor==1:    
         target_file.write("\nCorrected code recommendation:\n\n")
         corrections = Output1_dict.get('Corrections')

         if isinstance(corrections, list) and corrections:
            for item in corrections:
               explanation = item.get('Explanation', '')
               code = item.get('Correction', '')
               #target_file.write("Note: "+explanation + "\n")
               target_file.write(code + "\n\n")
         elif 'Correction' in Output1_dict:
            target_file.write(str(Output1_dict['Correction']))
      else:
         target_file.write("\nRefactored versions code:\n\n")
         target_file.write(Output1_dict['Refactored Versions'])
      target_file.write("\n\n")

      print(f"The output is saved to {Results}\n")
   print(f"Inference Time: {format_time(InferenceTime)}")
   print(f"Execution time per input token: {format_time(TimePerInputToken)}")
else:
   print("Error in reading the Mamba file.") 
# ----------------------------------------------------------------------------